In [ ]:
import requests
import json

url = "http://192.168.0.2:11434/api/chat"

payload = {
    "model": "llama31",
    "messages": [
        {"role": "user", "content": "스트리밍으로 한 글자씩 출력해줘"}
    ],
    "stream": True
}

with requests.post(url, json=payload, stream=True) as r:
    for line in r.iter_lines():
        if line:
            data = json.loads(line.decode("utf-8"))
            if "message" in data:
                print(data["message"]["content"], end="", flush=True)
# 조회하기
import chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS"
)

client = chromadb.HttpClient(
    host="localhost",
    port=8000
)

collection = client.get_or_create_collection(
    name="cnn1_docs",
    embedding_function=ef
)

results = collection.get()
results
# 추가하기
import chromadb
from chromadb.utils import embedding_functions
import requests
import json

# =========================
# 1️⃣ 임베딩 함수 정의
# =========================
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS"
)

# =========================
# 2️⃣ ChromaDB 서버 연결
# =========================
client = chromadb.HttpClient(
    host="localhost",
    port=8000
)

# =========================
# 3️⃣ 컬렉션 생성/가져오기
# =========================
collection = client.get_or_create_collection(
    name="cnn1_docs",
    embedding_function=ef
)

# =========================
# 4️⃣ 샘플 질문/답변 데이터 추가
# =========================
qa_data = [
    {"id": "q1", "question": "한국 경제 전망은?", "answer": "한국 경제는 올해 완만한 성장세를 보일 것으로 예상됩니다."},
    {"id": "q2", "question": "금리 인상 영향은?", "answer": "금리 인상은 대출 비용을 증가시켜 소비를 억제할 수 있습니다."},
    {"id": "q3", "question": "물가 상승률 전망은?", "answer": "물가 상승률은 정부 정책과 국제 유가에 따라 다소 변동할 것으로 보입니다."}
]

documents = []
ids = []

for item in qa_data:
    doc_text = f"Q: {item['question']}\nA: {item['answer']}"
    documents.append(doc_text)
    ids.append(item['id'])

# ef 객체로 임베딩 생성 후 컬렉션에 추가
collection.add(
    documents=documents,
    ids=ids
)

print("컬렉션 문서 수:", collection.count())
# =========================
# 질문
# =========================
user_question = "금리 인상이 경제에 미치는 영향은?"

# =========================
# ChromaDB에서 가장 유사한 문서 검색
# =========================
results = collection.query(
    query_texts=[user_question],  # ChromaDB가 ef로 임베딩 자동 생성
    n_results=1                  # 가장 유사한 1개 문서만
)

# 결과 출력
print(results)
# =========================
# 질문
# =========================
user_question = "금리 인상이 경제에 미치는 영향은?"

# =========================
# ChromaDB에서 가장 유사한 문서 검색
# =========================
results = collection.query(
    query_texts=[user_question],  # ChromaDB가 ef로 임베딩 자동 생성
    n_results=1                   # 가장 유사한 1개 문서만
)

# =========================
# 결과 확인
# =========================
similar_doc = results["documents"][0][0]
print("가장 유사한 문서:")
print(similar_doc)

def build_strict_prompt(user_question, context):
    """
    user_question: 사용자가 입력한 질문
    context: ChromaDB에서 검색된 가장 유사한 문서

    반환값: LLM에 바로 넣을 프롬프트
    """
    return f"""
        너는 아래 [문서]에 있는 내용만 사용해서 답변해야 한다.
        문서에 없는 내용이면 반드시 "해당 정보는 문서에 없습니다."라고 답변하라.

        [문서]
        {context}

        [질문]
        {user_question}
    """

# =========================
# 5️⃣ ChromaDB 검색 함수
# =========================
def retrieve_context(user_question, k=1):
    results = collection.query(
        query_texts=[user_question],
        n_results=k
    )
    context = results["documents"][0][0]  # 가장 유사한 1개 문서
    print(context)
    return context

# =========================
# 6️⃣ Strict 프롬프트 생성 함수
# =========================
def build_strict_prompt(user_question, context):
    return f"""
        너는 아래 [문서]에 있는 내용만 사용해서 답변해야 한다.
        문서에 없는 내용이면 반드시 "해당 정보는 문서에 없습니다."라고 답변하라.

        [문서]
        {context}

        [질문]
        {user_question}
    """

def build_flexible_prompt(user_question, context):
    """
    문서를 참고하되, LLM이 자유롭게 답변하도록 유도
    """
    return f"""
        아래 문서를 참고하여 질문에 답변해 주세요. 
        질문과 관련된 정보를 문서에서 찾고, 자연스럽게 설명해 주세요.

        [문서]
        {context}

        [질문]
        {user_question}
        """



# =========================
# 7️⃣ 스트리밍 LLM 호출 함수
# =========================
OLLAMA_URL = "http://192.168.0.2:11434/api/chat"
MODEL = "llama31"

def ask_llm_with_chroma(user_question):
    # 1️⃣ Chroma 검색
    context = retrieve_context(user_question)

    # 2️⃣ 프롬프트 생성
    prompt = build_flexible_prompt(user_question, context)

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": True
    }

    # 3️⃣ 스트리밍 요청
    with requests.post(OLLAMA_URL, json=payload, stream=True) as r:
        for line in r.iter_lines():
            if line:
                data = json.loads(line.decode("utf-8"))
                if "message" in data:
                    print(data["message"]["content"], end="", flush=True)

# =========================
# 8️⃣ 실행 예제
# =========================
question = "금리 인상 영향은?"
ask_llm_with_chroma(question)